In [1]:
import pandas as pd

In [2]:
df= pd.read_csv("data/customer_churn_eda_messy.csv")

In [3]:
df.head()

,customer_id,signup_date,last_active_date,age,region,city_raw,customer_segment,acquisition_channel,product_plan,device_type,monthly_sessions,avg_monthly_spend,discount_pct,support_tickets_90d,satisfaction_score,estimated_income_czk,payment_method,churned
0,100001,2025-01-31,2026-04-23,39.0,Prague,Praha,Family,Organic Search,Basic,desktop,9.0,633.59,13.9,3.0,7.8,99500.0,card,0
1,100002,2025-12-30,2026-04-27,53.0,Brno,Brno,Family,Email Campaign,Business,desktop,22.0,15337.56,4.2,2.0,5.9,68800.0,NaN,0
2,100003,2024-05-10,2026-04-20,27.0,Olomouc,olomouc,Family,Organic Search,Premium,tablet,9.0,3772.14,19.9,2.0,4.7,62600.0,card,0
3,100004,2025-07-18,2026-04-05,28.0,Brno,Brno,Small Business,Email Campaign,Plus,mobile,15.0,3633.56,4.8,0.0,8.6,54200.0,paypal,0
4,100005,2025-03-22,2026-04-09,54.0,Ostrava,Ostrava,Family,Social Media,Business,mobile,13.0,4665.13,12.8,2.0,10.0,38800.0,card,0


# Data cleaning

In [4]:
df.describe()

,customer_id,age,monthly_sessions,avg_monthly_spend,discount_pct,support_tickets_90d,satisfaction_score,estimated_income_czk,churned
count,1.208000e+04,12080.000000,12080.000000,12080.000000,12080.000000,11821.000000,11186.000000,11990.000000,12080.000000
mean,1.119584e+05,38.782119,8.826821,2104.408472,13.489694,1.626681,6.822689,50264.419766,0.036341
std,7.305425e+04,23.295168,5.265292,4299.859074,9.404996,1.343769,3.038383,36162.794290,0.187145
min,1.000010e+05,-5.000000,0.000000,-999.000000,0.000000,0.000000,0.000000,-1000.000000,0.000000
25%,1.030208e+05,29.000000,5.000000,449.602500,6.100000,1.000000,5.700000,29700.000000,0.000000
50%,1.060405e+05,38.000000,8.000000,1053.265000,11.900000,1.000000,6.800000,43800.000000,0.000000
75%,1.090602e+05,47.000000,12.000000,2355.172500,19.200000,2.000000,7.800000,62700.000000,0.000000
max,1.011749e+06,999.000000,39.000000,171115.300428,60.500000,11.000000,99.000000,999999.000000,1.000000


In [5]:
#  no duplicates
df.drop_duplicates().count()

customer_id             12080
signup_date             10332
last_active_date        12073
age                     12080
region                  11925
city_raw                12080
customer_segment        12080
acquisition_channel     11945
product_plan            11977
device_type             12080
monthly_sessions        12080
avg_monthly_spend       12080
discount_pct            12080
support_tickets_90d     11821
satisfaction_score      11186
estimated_income_czk    11990
payment_method          11595
churned                 12080
dtype: int64

## Handle missing values

In [6]:
# ~17% of signup_date, ~7% of satisfaction_score, 4% payment_method are missing
# ~1.1% of regions may be recoverable from city_raw
df.isna().sum() *100/df.count()

customer_id              0.000000
signup_date             16.918312
last_active_date         0.057981
age                      0.000000
region                   1.299790
city_raw                 0.000000
customer_segment         0.000000
acquisition_channel      1.130180
product_plan             0.859982
device_type              0.000000
monthly_sessions         0.000000
avg_monthly_spend        0.000000
discount_pct             0.000000
support_tickets_90d      2.191016
satisfaction_score       7.992133
estimated_income_czk     0.750626
payment_method           4.182837
churned                  0.000000
dtype: float64

In [7]:
# signup_date missingness is evenly distributed across churned/non-churned
# and shows no pattern with spend or sessions — likely missing at random
df.groupby(df["signup_date"].isna())["churned"].mean()
df.groupby(df["signup_date"].isna())["monthly_sessions"].mean()
df.groupby(df["signup_date"].isna())["avg_monthly_spend"].mean()
df[df["signup_date"].isna()]["acquisition_channel"].value_counts()


acquisition_channel
Organic Search    407
Paid Ads          356
Social Media      247
Email Campaign    209
Referral          209
Direct            202
Partner            97
Name: count, dtype: int64

In [8]:
# Non-churned customers are slightly more likely to have missing satisfaction scores (6.5% vs 3.4%)
# Missingness is not completely random — a flag column is added before imputation
df.groupby(df["satisfaction_score"].isna())["churned"].mean()
df["satisfaction_missing"] = df["satisfaction_score"].isna().astype(int)
df["satisfaction_score"] = df["satisfaction_score"].fillna(
    df.groupby("churned")["satisfaction_score"].transform("median")
)
df["satisfaction_score"].unique()

array([ 7.8,  5.9,  4.7,  8.6, 10. ,  4.3,  7.1,  5.7,  8.9,  6.8,  7.6,
        7.5,  8. ,  5.4,  5.5,  9.6,  6.4,  7. ,  6.3,  6.1,  7.7,  6.9,
        6.7,  5. ,  6. ,  6.2,  8.5,  8.7,  9.8,  6.6,  7.9,  9.4,  4.4,
        4.9,  8.2,  8.4,  9.2,  4.6,  7.2,  8.8,  5.6,  5.8,  7.4,  7.3,
        5.2,  5.3,  4.8,  4.5,  3.9,  8.3,  6.5,  3. ,  2.9,  9. ,  4. ,
        3.7,  2.4,  9.5,  9.3,  9.1,  8.1,  3.5,  2.2,  3.8,  4.2, 99. ,
        5.1,  3.2,  3.6,  4.1,  9.7,  9.9,  3.1,  3.3,  3.4,  2.7,  2.6,
        2. ,  1. ,  2.8,  1.8,  2.1,  1.7,  1.4,  1.5,  1.6,  0. ,  2.3,
       11. ,  1.9,  1.2,  2.5])

In [9]:
# Missing signup_date doesnt depend on churned status
df.groupby('churned')['signup_date'].apply(lambda x: x.isna().mean())*100

churned
0    14.466111
1    14.578588
Name: signup_date, dtype: float64

In [10]:
df["acquisition_channel"].unique()

<StringArray>
['Organic Search', 'Email Campaign',   'Social Media',       'Paid Ads',
       'Referral',         'Direct',              nan,        'Partner']
Length: 8, dtype: str

In [11]:
df["payment_method"] = df["payment_method"].fillna("unknown")
df["acquisition_channel"] = df["acquisition_channel"].fillna("unknown")

In [12]:
df=df.dropna(subset="estimated_income_czk")
df=df.dropna(subset="product_plan")

In [13]:
# Likely means no tickets
df["support_tickets_90d"] = df["support_tickets_90d"].fillna(0)

## Invalid values

In [14]:
# Removed 12 customers with negative age and 13 over 110 — likely data entry errors
df[df["age"]<0].count() 
df[df["age"]>110].count()
df = df[df["age"]>0]
df = df[df["age"]<110]

In [15]:
# Removed 7 customers with last_active_date before 1950 — invalid timestamps
df["last_active_date"] = pd.to_datetime(df["last_active_date"], errors="coerce")
df[df["last_active_date"].dt.year <1950].count()

df = df[df["last_active_date"].dt.year >1950]

In [16]:
# Removed 17 customers with negative avg_monthly_spend — invalid values
df[df["avg_monthly_spend"]<0].count()
df = df[df["avg_monthly_spend"]>0]

## Correct data types

In [17]:
df.dtypes

customer_id                      int64
signup_date                        str
last_active_date        datetime64[us]
age                            float64
region                             str
city_raw                           str
customer_segment                   str
acquisition_channel                str
product_plan                       str
device_type                        str
monthly_sessions               float64
avg_monthly_spend              float64
discount_pct                   float64
support_tickets_90d            float64
satisfaction_score             float64
estimated_income_czk           float64
payment_method                     str
churned                          int64
satisfaction_missing             int64
dtype: object

In [18]:
for col_name in df.columns:
    if (df[col_name].dtype == "str"):
        df[col_name] = df[col_name].str.strip().str.lower()

In [19]:
df["age"] = pd.to_numeric(df["age"]).astype("int64")
df["monthly_sessions"] = pd.to_numeric(df["monthly_sessions"]).astype("int64")
df["estimated_income_czk"] = pd.to_numeric(df["estimated_income_czk"], errors="coerce").astype("Int64")

In [20]:
df["payment_method"].drop_duplicates().sort_values(ascending=False)

1           unknown
3            paypal
62             cash
0              card
7     bank_transfer
Name: payment_method, dtype: str

In [21]:
df["support_tickets_90d"]=df["support_tickets_90d"].astype("Int32")

## Handle missing regions

In [22]:
df["city_raw"].value_counts()

city_raw
other      1464
prague     1453
ostrava    1225
brno       1162
olomouc     850
praha       751
prg         721
unknown     693
liberec     678
brn         635
plzen       338
plzeň       317
pilsen      292
Name: count, dtype: int64

In [23]:
df[df["city_raw"] == "unknown"]

,customer_id,signup_date,last_active_date,age,region,city_raw,customer_segment,acquisition_channel,product_plan,device_type,monthly_sessions,avg_monthly_spend,discount_pct,support_tickets_90d,satisfaction_score,estimated_income_czk,payment_method,churned,satisfaction_missing
15,100016,2023-11-05,2026-04-17,53,other,unknown,small business,organic search,plus,mobile,14,3507.01,20.2,1,5.5,64800,card,0,0
40,100041,2023-12-25,2026-04-04,49,other,unknown,young professional,organic search,plus,desktop,9,977.76,3.3,1,6.8,28500,bank_transfer,0,1
65,100066,2024-02-15,2026-04-02,31,other,unknown,family,social media,plus,mobile,7,2868.52,5.1,1,7.5,33900,card,0,0
67,100068,2022-07-25,2026-04-08,43,NaN,unknown,young professional,social media,plus,tablet,4,496.60,16.9,1,7.9,41200,card,0,0
96,100097,2025-03-28,2026-04-27,45,other,unknown,young professional,paid ads,plus,unknown,7,1431.71,20.5,0,6.8,32700,bank_transfer,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11955,111956,2023-06-14,2026-04-12,42,other,unknown,young professional,organic search,basic,desktop,8,1079.01,13.8,2,6.0,34800,card,0,0
11982,111983,2022-01-05,2026-03-12,37,other,unknown,young professional,paid ads,basic,desktop,13,2041.40,33.5,1,4.2,29600,bank_transfer,0,0
12060,1004476,2024-04-02,2026-04-20,56,other,unknown,student,referral,basic,mobile,6,525.61,20.8,1,6.1,20500,card,0,0
12061,1004107,2023-01-26,2026-03-26,57,other,unknown,family,organic search,premium,desktop,10,743.48,18.7,0,6.8,38800,bank_transfer,0,1


In [24]:
city_to_region = (
    df.dropna(subset=["region"])
      .drop_duplicates(subset=["city_raw"])
      .set_index("city_raw")["region"]
)

df["region"] = df["region"].fillna(df["city_raw"].map(city_to_region))
city_to_region

city_raw
praha       prague
brno          brno
olomouc    olomouc
ostrava    ostrava
plzen        plzen
liberec    liberec
unknown      other
other        other
prague      prague
plzeň        plzen
pilsen       plzen
brn           brno
prg         prague
Name: region, dtype: str

In [25]:
city_mapping = {
    "praha": "prague",
    "prg": "prague",
    "prague": "prague",

    "brno": "brno",
    "brn": "brno",

    "olomouc": "olomouc",

    "ostrava": "ostrava",

    "plzen": "plzen",
    "plzeň": "plzen",
    "pilsen": "plzen",

    "liberec": "liberec",

    "unknown": "unknown",
    
    "other": "other"
}

df["city_clean"] = df["city_raw"].replace(city_mapping)

In [26]:
df["city_clean"].unique()

<StringArray>
['prague', 'brno', 'olomouc', 'ostrava', 'plzen', 'liberec', 'unknown',
 'other']
Length: 8, dtype: str

In [27]:
#  after cleaning we got 10579 customers
df.describe()

,customer_id,last_active_date,age,monthly_sessions,avg_monthly_spend,discount_pct,support_tickets_90d,satisfaction_score,estimated_income_czk,churned,satisfaction_missing
count,1.057900e+04,10579,10579.000000,10579.000000,10579.000000,10579.000000,10579.0,10579.000000,10579.0,10579.000000,10579.000000
mean,1.119518e+05,2026-04-09 16:46:35.878627,38.431515,9.439550,2355.102216,13.555667,1.557992,6.853587,50220.889782,0.032895,0.064184
min,1.000010e+05,2025-03-01 00:00:00,18.000000,1.000000,28.280000,0.000000,0.0,0.000000,-1000.0,0.000000,0.000000
25%,1.030165e+05,2026-04-07 00:00:00,29.000000,6.000000,612.480000,6.100000,1.0,5.900000,29700.0,0.000000,0.000000
50%,1.060340e+05,2026-04-16 00:00:00,38.000000,9.000000,1229.050000,11.900000,1.0,6.800000,43600.0,0.000000,0.000000
75%,1.090645e+05,2026-04-22 00:00:00,47.000000,12.000000,2621.410000,19.300000,2.0,7.800000,62600.0,0.000000,0.000000
max,1.011749e+06,2027-05-10 00:00:00,82.000000,39.000000,171115.300428,60.500000,10.0,99.000000,999999.0,1.000000,1.000000
std,7.302168e+04,NaN,12.234911,4.879277,4502.674093,9.436984,1.332902,2.798636,37027.28395,0.178371,0.245092


# Save to CSV

In [28]:
df.to_csv("data/cleaned_data.csv", index=False)